# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gamal1osama/flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This notebook constructs the feature vector for Lane 2 (Refresh / Content Opportunity Scoring) and executes rigorous leakage, privacy, and boundary checks to ensure no target-derived or future signals enter the model.

## 1. Build the feature vector

### Feature Vector Construction
We construct a 60-column pre-decision feature matrix from `data/raw/content_refresh_anonymized.csv`:
- **Volume Signals (Log-Transformed):** `impressions_90d`, `clicks_90d`, `sessions_90d`, `search_volume` transformed via `log1p` to handle extreme heavy tails.
- **Search Presence:** `days_with_impressions`, `days_with_sessions` (0–90 days).
- **Efficiency & SERP Position:** `avg_position`, `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`.
- **Content Metrics:** `word_count`, `char_count`, `content_age_days`, `days_since_last_update`.
- **Categorical Encodings:** One-hot encoded `competition_level`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

data_candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank/data/raw/content_refresh_anonymized.csv"),
]
data_path = next((p for p in data_candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy().reset_index(drop=True)
eligible["is_declining"] = (eligible["trend_direction"] == "down").astype(int)

num_cols = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]
cat_cols = [
    "competition_level", "content_type", "main_intent", "age_tier", 
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"
]

X_num = eligible[num_cols].fillna(0)
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "search_volume"]:
    X_num["log_" + c] = np.log1p(X_num[c])

X_cat = pd.get_dummies(eligible[cat_cols].fillna("unknown").astype(str), prefix=cat_cols, dtype=float)
X_all = pd.concat([X_num, X_cat], axis=1)

print(f"Built Feature Matrix: {X_all.shape[0]:,} rows x {X_all.shape[1]} features")
print(f"Eligible URLs: {len(eligible):,} | Clients: {eligible['client_id'].nunique()}")
print(f"Target is_declining distribution: {eligible['is_declining'].mean():.1%} down")


Built Feature Matrix: 30,000 rows x 60 features
Eligible URLs: 30,000 | Clients: 32
Target is_declining distribution: 54.2% down


## 2. Feature notes (meaning, missing, categorical, available-when?)

### Feature Documentation & Missingness Audit
- **Missingness Strategy:** Missing keyword metrics (`search_volume`, `cpc`, `competition`) correlate with `content_type == 'feedly article'`. We impute numerical blanks with 0 and categorical blanks with `'unknown'`. Categorical one-hot encoding explicitly preserves missingness signals without leakage.
- **Temporal Availability:** All selected features describe trailing 90-day activity ending at the decision cutoff. Every feature is fully knowable *before* the forward outcome window begins.

In [2]:
# Check missing values and data availability
print("=== Missingness Audit on Raw Numerical Inputs ===")
missing_stats = df[num_cols].isna().sum()
missing_report = pd.DataFrame({
    "missing_count": missing_stats[missing_stats > 0],
    "missing_pct": (missing_stats[missing_stats > 0] / len(df) * 100).round(2)
})
print(missing_report)

print("\nZero NaNs in final engineered feature matrix:", X_all.isna().sum().sum() == 0)


=== Missingness Audit on Raw Numerical Inputs ===
               missing_count  missing_pct
search_volume           2468         8.23
competition             2468         8.23
cpc                     2468         8.23
word_count              7699        25.66
char_count              7699        25.66
scroll_rate              125         0.42

Zero NaNs in final engineered feature matrix: True


## 3. The leakage hunt

### Leakage Attack & Integrity Checks
We systematically test for three classic leakage vectors:
1. **Label Leakage:** Ensuring `trend_direction` and `trend_pct` never enter the feature matrix.
2. **Outcome Overlap:** Confirming no 30-day outcome window columns (`impressions_last_30d`, etc.) are included.
3. **Domain Authority Leakage:** Ensuring client identifiers are strictly quarantined for grouped validation and never used as features.

In [3]:
# Leakage Attack: Assert quarantined columns are absent
banned_columns = [
    "trend_direction", "trend_pct", "is_declining", "is_declining_label",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "client_id", "content_id"
]

print("=== Leakage Assertions ===")
for col in banned_columns:
    assert col not in X_all.columns, f"CRITICAL LEAKAGE DETECTED: {col} in feature matrix!"
    print(f"✓ Verified absent: {col}")

# Verify maximum feature correlation with target
corrs = X_all.apply(lambda col: np.corrcoef(col, eligible["is_declining"])[0, 1]).abs()
max_corr_feature = corrs.idxmax()
max_corr_val = corrs.max()
print(f"\nMaximum absolute feature correlation with target: {max_corr_feature} (r = {max_corr_val:.3f})")
assert max_corr_val < 0.85, f"Suspiciously high feature correlation detected: {max_corr_val}"
print("✓ No suspiciously high single-feature correlation. Leakage hunt passed!")


=== Leakage Assertions ===
✓ Verified absent: trend_direction
✓ Verified absent: trend_pct
✓ Verified absent: is_declining
✓ Verified absent: is_declining_label
✓ Verified absent: impressions_last_30d
✓ Verified absent: clicks_last_30d
✓ Verified absent: sessions_last_30d
✓ Verified absent: impressions_prev_30d
✓ Verified absent: clicks_prev_30d
✓ Verified absent: sessions_prev_30d
✓ Verified absent: client_id
✓ Verified absent: content_id

Maximum absolute feature correlation with target: days_with_impressions (r = 0.190)
✓ No suspiciously high single-feature correlation. Leakage hunt passed!


## 4. What I excluded and why

### Quarantined Fields and Exclusions
1. **`trend_direction` & `trend_pct`:** Excluded because the target label `is_declining` is derived directly from `trend_direction == 'down'`. Using them would be circular label leakage.
2. **`impressions_last_30d` & `clicks_last_30d`:** Excluded because they overlap the post-decision outcome window.
3. **`client_id` & `content_id`:** Excluded because pseudonymous identifiers allow models to memorize domain-specific baselines rather than learning generalizable search signals.
4. **`provider_used` & `model_used`:** Excluded because they are missing for $>75\%$ of content items and correlate with content generation batch dates rather than search engine demand.

In [4]:
# Print quarantined columns summary
exclusions = [
    ("trend_direction / trend_pct", "Direct target label source (circular leakage)"),
    ("impressions_last_30d / clicks_last_30d", "Outcome evaluation window (temporal leakage)"),
    ("client_id / content_id", "Pseudonymous identifiers (memorization risk; grouping only)"),
    ("provider_used / model_used", "Sparse metadata (75%+ missing, batch confounder)")
]
exclusion_df = pd.DataFrame(exclusions, columns=["Quarantined Field", "Exclusion Rationale"])
print("=== Quarantined Fields Summary ===")
print(exclusion_df.to_string(index=False))


=== Quarantined Fields Summary ===
                     Quarantined Field                                         Exclusion Rationale
           trend_direction / trend_pct               Direct target label source (circular leakage)
impressions_last_30d / clicks_last_30d                Outcome evaluation window (temporal leakage)
                client_id / content_id Pseudonymous identifiers (memorization risk; grouping only)
            provider_used / model_used            Sparse metadata (75%+ missing, batch confounder)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.